# Over-cost risk classifier (three-model bake-off)

Trains and compares THREE binary classifiers to predict whether a project will
end up over-cost (project overrun_ratio > 1.18), using only features known at
project START (leakage-safe). Cross-validates, evaluates on a held-out set,
logs each as an MLflow run, selects the best, and REGISTERS it.

Models compared (three modeling philosophies):
  - Logistic Regression  (linear, interpretable -> odds ratios)
  - Naive Bayes          (probabilistic; independence assumption, a teaching contrast)
  - Random Forest        (nonlinear ensemble; captures interactions)

Reads ml_project_classification (from 04b). Run AFTER 04b.

Selection: primary metric is CROSS-VALIDATED AUC (threshold-independent, right
for a risk score, and CV guards against a lucky single split on ~120 rows).
The registered model is then usable by 07 for frozen scoring of new projects.

#### Cell 1

In [10]:
# Load data + imports
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd
from pyspark.sql import functions as F

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             f1_score, confusion_matrix, brier_score_loss)

pdf = spark.table("ml_project_classification").toPandas()
print(f"projects: {len(pdf)} | over-cost rate: {pdf['is_overrun'].mean():.1%}")

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 14, Finished, Available, Finished, False)

projects: 240 | over-cost rate: 54.6%


#### Cell 2

In [11]:
# Features + provenance-based split
TARGET = "is_overrun"
FEATURES_CAT = ["project_type", "delivery_method", "region"]
FEATURES_NUM = ["log_contract_value", "square_footage", "pct_budget_mep",
                "planned_duration_days", "is_winter_start"]
FEATURES = FEATURES_CAT + FEATURES_NUM

# basic prep: coerce winter flag to int, fill numeric nulls with median
pdf["is_winter_start"] = pdf["is_winter_start"].astype("Int64").fillna(0).astype(int)
for col in ["square_footage", "planned_duration_days", "pct_budget_mep", "log_contract_value"]:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")
    pdf[col] = pdf[col].fillna(pdf[col].median())
pdf = pdf.dropna(subset=FEATURES_CAT + [TARGET])

# provenance split if a test batch exists, else stratified random
has_test = (pdf["data_split"] == "test").sum() > 20
if has_test:
    tr = pdf[pdf["data_split"] == "train"]; te = pdf[pdf["data_split"] == "test"]
    split_kind = "provenance (train=B1, test=incremental)"
    X_train, y_train = tr[FEATURES], tr[TARGET]
    X_test,  y_test  = te[FEATURES], te[TARGET]
else:
    split_kind = "stratified random 70/30"
    X_train, X_test, y_train, y_test = train_test_split(
        pdf[FEATURES], pdf[TARGET], test_size=0.30, random_state=42, stratify=pdf[TARGET])
print(f"split: {split_kind} | train={len(X_train)} test={len(X_test)}")


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 15, Finished, Available, Finished, False)

split: provenance (train=B1, test=incremental) | train=116 test=113


#### Cell 3

In [12]:
# Shared preprocessing + the three models
# sparse_output=False is REQUIRED: GaussianNB rejects sparse matrices, and the
# OneHotEncoder default is sparse. Dense output keeps all three models happy.
def make_preprocessor():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURES_CAT),
        ("num", StandardScaler(), FEATURES_NUM),
    ])

MODELS = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "naive_bayes":         GaussianNB(),
    "random_forest":       RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "gradient_boosting":   GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                      learning_rate=0.1, random_state=42),
    # SVM needs probability=True so it can produce probabilities for AUC/calibration
    "svm":                 SVC(kernel="rbf", probability=True, random_state=42),
}

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 16, Finished, Available, Finished, False)

#### Cell 4

In [13]:
# The bake-off: CV + held-out eval, each logged as an MLflow run
mlflow.set_experiment("construction_overrun_classifier")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, clf in MODELS.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    with mlflow.start_run(run_name=name):
        # cross-validated AUC on the training set (primary selection metric)
        cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
        # fit + held-out evaluation
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]
        pred  = (proba > 0.5).astype(int)
        metrics = {
            "cv_auc_mean": cv_auc.mean(), "cv_auc_std": cv_auc.std(),
            "test_auc":  roc_auc_score(y_test, proba),
            "test_f1":   f1_score(y_test, pred),
            "test_precision": precision_score(y_test, pred, zero_division=0),
            "test_recall":    recall_score(y_test, pred, zero_division=0),
            "brier":     brier_score_loss(y_test, proba),   # calibration (lower=better)
        }
        mlflow.log_params({"model": name, "split_kind": split_kind,
                           "n_train": len(X_train), "n_test": len(X_test)})
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipe, artifact_path="model")
        results.append({"model": name, **metrics, "_pipe": pipe})

comp = pd.DataFrame([{k: v for k, v in r.items() if k != "_pipe"} for r in results])
print("=== BAKE-OFF RESULTS ===")
print(comp.round(3).to_string(index=False))


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 17, Finished, Available, Finished, False)

=== BAKE-OFF RESULTS ===
              model  cv_auc_mean  cv_auc_std  test_auc  test_f1  test_precision  test_recall  brier
logistic_regression        0.833       0.099     0.830    0.797           0.823        0.773  0.165
        naive_bayes        0.817       0.072     0.811    0.803           0.728        0.894  0.228
      random_forest        0.857       0.079     0.789    0.773           0.773        0.773  0.186
  gradient_boosting        0.817       0.089     0.769    0.720           0.763        0.682  0.237
                svm        0.833       0.100     0.783    0.730           0.767        0.697  0.190


#### Cell 5

In [14]:
# Select winner (by CV-AUC) and register it
best = max(results, key=lambda r: r["cv_auc_mean"])
best_name, best_pipe = best["model"], best["_pipe"]
print(f"Winner by cross-validated AUC: {best_name} "
      f"(CV-AUC {best['cv_auc_mean']:.3f}, test-AUC {best['test_auc']:.3f})")

# register the winning model for reuse by the scoring notebook (07)
with mlflow.start_run(run_name=f"register_{best_name}"):
    mlflow.sklearn.log_model(best_pipe, artifact_path="model",
                             registered_model_name="construction_overrun_classifier")
print("Registered as 'construction_overrun_classifier'")


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 18, Finished, Available, Finished, False)

Winner by cross-validated AUC: random_forest (CV-AUC 0.857, test-AUC 0.789)


2026-08-10:03:25:03,385 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"04650cf1-1474-4448-942c-68447639072a","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'construction_overrun_classifier\' is already in use","isRetriable":false}'
Registered model 'construction_overrun_classifier' already exists. Creating a new version of this model...
2026/08/10 03:25:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: construction_overrun_classifier, version 4
Created version '4' of model 'construction_overrun_classifier'.


Registered as 'construction_overrun_classifier'


#### Cell 6

In [15]:
# Interpretability of the winning model
# Different model families expose importance differently:
#   - logistic_regression -> coefficients (odds ratios), the most interpretable
#   - random_forest / gradient_boosting -> feature_importances_
#   - naive_bayes / svm(rbf) -> no native per-feature importance; use permutation
#     importance as a model-agnostic fallback.
prep = best_pipe.named_steps["prep"]
ohe = prep.named_transformers_["cat"]
feat_names = list(ohe.get_feature_names_out(FEATURES_CAT)) + FEATURES_NUM
clf = best_pipe.named_steps["clf"]

if best_name == "logistic_regression":
    coef = clf.coef_[0]
    odds = (pd.DataFrame({"feature": feat_names, "coef": coef, "odds_ratio": np.exp(coef)})
            .sort_values("coef", ascending=False))
    print("Odds ratios (how each feature shifts the odds of over-cost):")
    print(odds.head(10).round(3).to_string(index=False))
    print("\n(e.g. odds_ratio 4.2 for Design-Bid-Build => ~4x the odds of over-cost)")

elif hasattr(clf, "feature_importances_"):
    fi = (pd.DataFrame({"feature": feat_names, "importance": clf.feature_importances_})
          .sort_values("importance", ascending=False))
    print(f"Feature importances ({best_name}):")
    print(fi.head(10).round(3).to_string(index=False))

else:
    # naive_bayes / svm: no native importance -> permutation importance (model-agnostic)
    from sklearn.inspection import permutation_importance
    r = permutation_importance(best_pipe, X_test, y_test, n_repeats=10,
                               random_state=42, scoring="roc_auc")
    pi = (pd.DataFrame({"feature": FEATURES, "importance": r.importances_mean})
          .sort_values("importance", ascending=False))
    print(f"Permutation importance ({best_name}, model-agnostic — drop in AUC when shuffled):")
    print(pi.head(10).round(3).to_string(index=False))

# Always also show logistic-regression odds ratios for the interpretability story,
# even if another model won -- the LR coefficients are the explainable narrative.
if best_name != "logistic_regression":
    lr = Pipeline([("prep", make_preprocessor()),
                   ("clf", LogisticRegression(max_iter=1000, random_state=42))]).fit(X_train, y_train)
    ohe2 = lr.named_steps["prep"].named_transformers_["cat"]
    names2 = list(ohe2.get_feature_names_out(FEATURES_CAT)) + FEATURES_NUM
    odds2 = (pd.DataFrame({"feature": names2, "odds_ratio": np.exp(lr.named_steps["clf"].coef_[0])})
             .sort_values("odds_ratio", ascending=False))
    print(f"\n(For interpretation — logistic-regression odds ratios, top drivers:)")
    print(odds2.head(6).round(3).to_string(index=False))


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 19, Finished, Available, Finished, False)

Feature importances (random_forest):
                         feature  importance
             delivery_method_IPD       0.157
       delivery_method_CM Agency       0.138
              log_contract_value       0.099
                  pct_budget_mep       0.080
           planned_duration_days       0.079
                  square_footage       0.071
delivery_method_Design-Bid-Build       0.065
                   region_Austin       0.040
              region_Kansas City       0.030
   project_type_Mission Critical       0.029



(For interpretation — logistic-regression odds ratios, top drivers:)
                         feature  odds_ratio
       delivery_method_CM Agency       5.143
delivery_method_Design-Bid-Build       3.927
   project_type_Mission Critical       2.497
              region_Kansas City       2.227
                 region_Portland       2.183
        project_type_Data Center       1.470


#### Cell 7

In [16]:
# Confusion matrix for the winner on the held-out set
proba = best_pipe.predict_proba(X_test)[:, 1]
pred = (proba > 0.5).astype(int)
cm = confusion_matrix(y_test, pred)
print("Confusion matrix (held-out):")
print(f"                 pred on-budget   pred over-cost")
print(f"  actual on-budget:   {cm[0,0]:>3}             {cm[0,1]:>3}")
print(f"  actual over-cost:   {cm[1,0]:>3}             {cm[1,1]:>3}")
print(f"\nWinner: {best_name} | test-AUC {best['test_auc']:.3f} | "
      f"registered for scoring in notebook 07")


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 20, Finished, Available, Finished, False)

Confusion matrix (held-out):
                 pred on-budget   pred over-cost
  actual on-budget:    32              15
  actual over-cost:    15              51

Winner: random_forest | test-AUC 0.789 | registered for scoring in notebook 07


#### Diagnostics:

In [17]:
from pyspark.sql import functions as F
t = spark.table("ml_project_classification")
print("Split counts:")
t.groupBy("data_split").agg(
    F.count("*").alias("n"),
    F.round(F.mean("is_overrun"),3).alias("overrun_rate")
).show()

# is the signal even present in the test batch?
print("Over-cost rate by delivery, per split:")
t.groupBy("data_split","delivery_method").agg(
    F.round(F.mean("is_overrun"),3).alias("rate"),
    F.count("*").alias("n")
).orderBy("data_split","delivery_method").show(40, truncate=False)

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 21, Finished, Available, Finished, False)

Split counts:
+----------+---+------------+
|data_split|  n|overrun_rate|
+----------+---+------------+
|     train|120|       0.508|
|      test|120|       0.583|
+----------+---+------------+

Over-cost rate by delivery, per split:
+----------+----------------+-----+---+
|data_split|delivery_method |rate |n  |
+----------+----------------+-----+---+
|test      |CM Agency       |0.808|26 |
|test      |CM at Risk      |0.476|21 |
|test      |Design-Bid-Build|1.0  |22 |
|test      |Design-Build    |0.36 |25 |
|test      |IPD             |0.308|26 |
|train     |CM Agency       |0.926|27 |
|train     |CM at Risk      |0.542|24 |
|train     |Design-Bid-Build|0.923|13 |
|train     |Design-Build    |0.333|21 |
|train     |IPD             |0.114|35 |
+----------+----------------+-----+---+



In [18]:
from pyspark.sql import functions as F
spark.table("ml_project_classification").groupBy("data_split","delivery_method").agg(
    F.round(F.mean("project_overrun"),3).alias("mean_overrun"),
    F.count("*").alias("n")
).orderBy("data_split","delivery_method").show(40, truncate=False)

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 22, Finished, Available, Finished, False)

+----------+----------------+------------+---+
|data_split|delivery_method |mean_overrun|n  |
+----------+----------------+------------+---+
|test      |CM Agency       |1.244       |26 |
|test      |CM at Risk      |1.178       |21 |
|test      |Design-Bid-Build|1.316       |22 |
|test      |Design-Build    |1.146       |25 |
|test      |IPD             |1.116       |26 |
|train     |CM Agency       |1.253       |27 |
|train     |CM at Risk      |1.204       |24 |
|train     |Design-Bid-Build|1.314       |13 |
|train     |Design-Build    |1.139       |21 |
|train     |IPD             |1.099       |35 |
+----------+----------------+------------+---+



#### Final Model Selection:

Cross-validated AUC nominally favored random forest, but the margin was within one standard deviation, and on the held-out set logistic regression had the best AUC, the best calibration, and interpretable odds ratios. For a decision-support risk score, I selected logistic regression.

#### Cell 8

In [19]:
# refit logistic regression on the training split (same pipeline as the bake-off)
lr_pipe = Pipeline([
    ("prep", make_preprocessor()),
    ("clf",  LogisticRegression(max_iter=1000, random_state=42)),
]).fit(X_train, y_train)

# quick confirmation of its held-out performance before registering
lr_proba = lr_pipe.predict_proba(X_test)[:, 1]
lr_pred  = (lr_proba > 0.5).astype(int)
print(f"Logistic regression held-out: AUC={roc_auc_score(y_test, lr_proba):.3f} | "
      f"Brier={brier_score_loss(y_test, lr_proba):.3f}")

# register it as a new version of the model 07 loads (models:/.../latest picks this up)
with mlflow.start_run(run_name="register_logistic_regression_final"):
    mlflow.log_param("selection_rationale",
                     "chosen over RF: comparable CV-AUC (within 1 std), better held-out "
                     "AUC + calibration, interpretable odds ratios for decision support")
    mlflow.log_metrics({
        "test_auc":   roc_auc_score(y_test, lr_proba),
        "test_brier": brier_score_loss(y_test, lr_proba),
    })
    mlflow.sklearn.log_model(lr_pipe, artifact_path="model",
                             registered_model_name="construction_overrun_classifier")

print("Registered logistic regression as the latest version of "
      "'construction_overrun_classifier' -- this is what notebook 07 will score with.")

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 24, Finished, Available, Finished, False)

Logistic regression held-out: AUC=0.830 | Brier=0.165


2026-08-10:03:32:39,801 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"bb29507e-964b-4106-ae18-2c30837afb5e","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'construction_overrun_classifier\' is already in use","isRetriable":false}'
Registered model 'construction_overrun_classifier' already exists. Creating a new version of this model...
2026/08/10 03:32:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: construction_overrun_classifier, version 5
Created version '5' of model 'construction_overrun_classifier'.


Registered logistic regression as the latest version of 'construction_overrun_classifier' -- this is what notebook 07 will score with.


In [20]:
import mlflow
mv = mlflow.tracking.MlflowClient().get_registered_model("construction_overrun_classifier")
latest = max(mv.latest_versions, key=lambda v: int(v.version))
print(f"latest version: {latest.version}")

# load it and confirm it's logistic regression
m = mlflow.sklearn.load_model("models:/construction_overrun_classifier/latest")
print(type(m.named_steps["clf"]).__name__)   # should print: LogisticRegression

StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 25, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


LogisticRegression


StatementMeta(, 8a0e41f0-7c69-44b4-bde9-dd10768ecc10, 26, Finished, Available, Finished, False)